## Intitutional Holdings from LSEG (Thomson-Reuters) Institutional (13F) Holdings (S34)

Rong Wang, June 2026

(Code adapted from [Qingyi (Freda) Drechsler](https://www.fredasongdrechsler.com/data-crunching/io_breadth)'s original implementation).

In [1]:
# Packages
import pandas as pd
import numpy as np
import wrds
import datetime as dt
from pandas.tseries.offsets import *
import matplotlib.pyplot as plt
from pathlib import Path

# Define directories
datapath = Path("/work/rw196/holdings")

# WRDS connection
conn = wrds.Connection(wrds_username='adamwang08')

Loading library list...
Done


In [2]:
# Sample period
startdate = '03/01/1980'
enddate   = '12/31/2025'

##### Notice the transition from CRSP SIZ/FIZ to CIZ Format. 

CRSP has migrated its U.S. Stock Database from the legacy SIZ/FIZ format to the new CIZ (CRSP Integrated Zip) format. The CIZ architecture is designed to consolidate information that previously resided in multiple files, improve consistency across CRSP products, provide richer security-level metadata, and simplify common research workflows by reducing the number of auxiliary merges required. As part of this transition, the legacy stock files (e.g., msf, dsf) are no longer the primary production datasets and are no longer updated after the December 2024 release. New CRSP releases are distributed through the CIZ framework and exposed in WRDS through tables such as msf_v2 and related security master datasets.es.

Because of this transition, several aspects of the conventional CRSP workflow must be modified. Detailed discussions of the SIZ-to-CIZ migration, variable mappings, and methodological differences can be found in the CRSP and WRDS documentation:

- [WRDS CRSP Data Changes Announcement](https://wrds-www.wharton.upenn.edu/pages/data-announcements/changes-to-crsp-data/)
- [CRSP US Stock & Indexes Database Guide: Flat File Format 2.0 (CIZ)](https://wrds-www.wharton.upenn.edu/documents/1996/CRSP_US_Stock__Indexes_Database_Guide_Flat_File_Format_2.0.pdf)
- [CRSP SIZ-to-CIZ Cross-Reference Guide](https://www.crsp.org/wp-content/uploads/guides/CRSP_Cross_Reference_Guide_1.0_to_2.0.pdf)

The main changes in this file are:

1. Security identifier mapping: The conventional approach uses the legacy crsp.msenames table to map securities between CUSIPs and PERMNOs. Here, I instead use crsp.stocknames_v2, which supersedes the legacy name files like crsp.msenames, crsp.dsenames, and older crsp.stocknames).

2. Monthly stock data and common-stock identification: The conventional approach relies on the legacy monthly stock file crsp.msf, where common stocks are typically identified using CRSP share codes (shrcd = 10 or 11). In this project, I use the CIZ-based monthly stock file crsp.msf_v2, which supersedes crsp.msf and continues to receive updates beyond December 2024. Under the CIZ framework, the legacy share-code classification is replaced by a richer security classification system. Consequently, common stocks are identified using a combination of security characteristics, including sharetype, securitytype, securitysubtype, usincflg, and issuertype, following the mappings recommended by CRSP and WRDS to replicate the traditional common-stock universe.

These changes preserve the economic interpretation of the sample while aligning the data construction process with the current CRSP database structure and ensuring compatibility with future data releases.

### Step 1: LSEG (Thomson-Reuters) Institutional (13F) Holdings (S34)

Type 1: Manager, first vintage

In [3]:
%%time
fst_vint = conn.raw_sql("""
                      select rdate, fdate, mgrno, mgrname
                      from tfn.s34type1 
                      """, date_cols=['rdate','fdate']) 

# Keep first vintage with holding data for each mgrno-rdate combo
min_fdate = fst_vint.groupby(['mgrno','rdate'])['fdate'].min().reset_index()

# Merge back with the fst_vint data to keep only the first vintage records
fst_vint = pd.merge(fst_vint, min_fdate, how='inner', on=['mgrno','rdate','fdate'])

# Sort by mgrno and rdate and create lag_rdate to calculate gap
fst_vint = fst_vint.sort_values(['mgrno', 'rdate'])
fst_vint['lag_rdate']=fst_vint.groupby(['mgrno'])['rdate'].shift(1)

# Number of quarters gap between rdate and lag_rdate
fst_vint['rdate_year']=fst_vint.rdate.dt.year
fst_vint['rdate_qtr'] =fst_vint.rdate.dt.quarter
fst_vint['lag_rdate_year']=fst_vint.lag_rdate.dt.year
fst_vint['lag_rdate_qtr'] =fst_vint.lag_rdate.dt.quarter
fst_vint['qtr'] = (fst_vint.rdate_year - fst_vint.lag_rdate_year)*4 + (fst_vint.rdate_qtr - fst_vint.lag_rdate_qtr)

# label first_report flag
fst_vint['first_report'] = ((fst_vint.qtr.isnull()) | (fst_vint.qtr>=2))
fst_vint = fst_vint.drop(['qtr'],axis=1)
fst_vint

CPU times: user 3.04 s, sys: 208 ms, total: 3.25 s
Wall time: 3.82 s


,rdate,fdate,mgrno,mgrname,lag_rdate,rdate_year,rdate_qtr,lag_rdate_year,lag_rdate_qtr,first_report
68889,1998-12-31,1998-12-31,110.0,A R ASSET MANAGEMENT INC,NaT,1998,4,NaN,NaN,True
70538,1999-03-31,1999-03-31,110.0,A R ASSET MANAGEMENT INC,1998-12-31,1999,1,1998.0,4.0,False
72009,1999-06-30,1999-06-30,110.0,"A R ASSET MANAGEMENT, INC.",1999-03-31,1999,2,1999.0,1.0,False
73515,1999-09-30,1999-09-30,110.0,"A R ASSET MANAGEMENT, INC.",1999-06-30,1999,3,1999.0,2.0,False
75107,1999-12-31,1999-12-31,110.0,"A R ASSET MANAGEMENT, INC.",1999-09-30,1999,4,1999.0,3.0,False
...,...,...,...,...,...,...,...,...,...,...
34595,1991-09-30,1991-09-30,95120.0,ZWEIG TOTAL RETURN ADVS.,1991-06-30,1991,3,1991.0,2.0,False
35618,1991-12-31,1991-12-31,95120.0,ZWEIG TOTAL RETURN ADVS.,1991-09-30,1991,4,1991.0,3.0,False
36682,1992-03-31,1992-03-31,95120.0,ZWEIG TOTAL RETURN ADVS.,1991-12-31,1992,1,1991.0,4.0,False
37734,1992-06-30,1992-06-30,95120.0,ZWEIG TOTAL RETURN ADVS.,1992-03-31,1992,2,1992.0,1.0,False


In [4]:
# Last report by manager or missing 13F reports in the next quarter(s)
fst_vint = fst_vint.sort_values(['mgrno','rdate'], ascending=[True, False])
fst_vint['lead_rdate']=fst_vint.groupby(['mgrno'])['rdate'].shift(1)

# Number of quarters gap between lead_rdate and rdate
fst_vint['lead_rdate_year']=fst_vint.lead_rdate.dt.year
fst_vint['lead_rdate_qtr'] =fst_vint.lead_rdate.dt.quarter
fst_vint['qtr'] = (fst_vint.lead_rdate_year - fst_vint.rdate_year)*4 + (fst_vint.lead_rdate_qtr - fst_vint.rdate_qtr)

# label last_report flag
fst_vint['last_report'] = ((fst_vint.qtr.isnull()) | (fst_vint.qtr>=2))
fst_vint = fst_vint.drop(['qtr'],axis=1)
fst_vint = fst_vint[(fst_vint['rdate']<=enddate) & (fst_vint['rdate']>=startdate)].drop(['lag_rdate','lead_rdate'], axis=1)

# Add total number of 13f filers during each quarter
fst_vint = fst_vint.sort_values(['rdate', 'mgrno'])
NumInst = fst_vint.groupby(['rdate'])['mgrno'].count().reset_index().rename(columns={'mgrno':'NumInst'})
fst_vint = pd.merge(fst_vint, NumInst, how='left', on='rdate')
fst_vint = fst_vint.drop(['mgrname'],axis=1)
fst_vint

,rdate,fdate,mgrno,rdate_year,rdate_qtr,lag_rdate_year,lag_rdate_qtr,first_report,lead_rdate_year,lead_rdate_qtr,last_report,NumInst
0,1980-03-31,1980-03-31,260.0,1980,1,NaN,NaN,True,1980.0,2.0,False,467
1,1980-03-31,1980-03-31,520.0,1980,1,NaN,NaN,True,1980.0,2.0,False,467
2,1980-03-31,1980-03-31,560.0,1980,1,NaN,NaN,True,1980.0,2.0,False,467
3,1980-03-31,1980-03-31,835.0,1980,1,NaN,NaN,True,1980.0,2.0,False,467
4,1980-03-31,1980-03-31,885.0,1980,1,NaN,NaN,True,1980.0,2.0,False,467
...,...,...,...,...,...,...,...,...,...,...,...,...
492163,2025-12-31,2025-12-31,94230.0,2025,4,2025.0,3.0,False,NaN,NaN,True,8713
492164,2025-12-31,2025-12-31,94280.0,2025,4,2025.0,3.0,False,NaN,NaN,True,8713
492165,2025-12-31,2025-12-31,94300.0,2025,4,2025.0,3.0,False,NaN,NaN,True,8713
492166,2025-12-31,2025-12-31,94500.0,2025,4,2025.0,3.0,False,NaN,NaN,True,8713


Type 3: Stock Holdings (it takes ~ 20 mins to load the dataset, s34type3)

In [ ]:
# %%time
# s34type3 = conn.raw_sql("""
#                       select fdate, mgrno, cusip, shares
#                       from tfn.s34type3
#                       """, date_cols=['fdate']) 

In [5]:
%%time
# s34type3.to_parquet(datapath / 's34type3.parquet', engine='pyarrow')
s34type3 = pd.read_parquet(datapath / 's34type3.parquet')

CPU times: user 17.8 s, sys: 7.07 s, total: 24.9 s
Wall time: 15 s


In [6]:
%%time
# Merge managers with holdings
holdings_v1 = pd.merge(fst_vint, s34type3, how='inner', on=['fdate','mgrno'] )
holdings_v1 = holdings_v1.drop(['rdate_year','rdate_qtr','lag_rdate_year','lag_rdate_qtr','first_report','lead_rdate_year','lead_rdate_qtr','last_report'], axis=1)
holdings_v1

CPU times: user 32.7 s, sys: 7.83 s, total: 40.5 s
Wall time: 40.6 s


,rdate,fdate,mgrno,NumInst,cusip,shares
0,1980-03-31,1980-03-31,260.0,467,00080010,50000.0
1,1980-03-31,1980-03-31,260.0,467,01371610,60000.0
2,1980-03-31,1980-03-31,260.0,467,01951910,70000.0
3,1980-03-31,1980-03-31,260.0,467,02687410,68250.0
4,1980-03-31,1980-03-31,260.0,467,02742910,45000.0
...,...,...,...,...,...,...
124054804,2025-12-31,2025-12-31,95105.0,8713,G9611510,184250.0
124054805,2025-12-31,2025-12-31,95105.0,8713,K0858810,15500.0
124054806,2025-12-31,2025-12-31,95105.0,8713,L8681T10,28235.0
124054807,2025-12-31,2025-12-31,95105.0,8713,M2682V10,65459.0


### Step 2: CRSP

#### Map cusips to permnos

In [7]:
# Get permno and start/end dates
crsp_names = conn.raw_sql("""
    select distinct permno, cusip, namedt, nameenddt
    from crsp.stocknames_v2
    where cusip != ''
""")
crsp_names['namedt'] = pd.to_datetime(crsp_names['namedt'])
crsp_names['namedt'] = pd.to_datetime(crsp_names['namedt'])

# Collapse of CRSP name history at the (permno, cusip) level, compressing multiple time intervals into one union
crsp_names = (
    crsp_names
    .groupby(["permno", "cusip"], as_index=False)
    .agg(
        namedt=("namedt", "min"),
        nameenddt=("nameenddt", "max")
    )
)

crsp_names = crsp_names.sort_values(by=['permno', 'cusip']).reset_index(drop=True)
crsp_names

,permno,cusip,namedt,nameenddt
0,10000,68391610,1986-01-07,1987-06-11
1,10001,29269V10,2009-08-04,2010-07-08
2,10001,29274A10,1993-11-22,2008-02-04
3,10001,29274A20,2008-02-05,2009-08-03
4,10001,36720410,2010-07-09,2017-08-03
...,...,...,...,...
55041,93433,92870X30,2013-04-24,2016-12-22
55042,93434,78513510,2010-06-14,2024-10-17
55043,93434,78513530,2024-10-18,2025-08-01
55044,93435,82936G20,2010-06-14,2012-05-18


In [8]:
%%time
# Map cusip to permno
holdings_v2 = holdings_v1.merge(crsp_names, how='inner', on='cusip')
holdings_v2

CPU times: user 58.4 s, sys: 8.64 s, total: 1min 7s
Wall time: 1min 7s


,rdate,fdate,mgrno,NumInst,cusip,shares,permno,namedt,nameenddt
0,1980-03-31,1980-03-31,260.0,467,00080010,50000.0,10006,1968-01-02,1984-06-28
1,1980-03-31,1980-03-31,260.0,467,01371610,60000.0,24264,1968-01-02,2007-11-14
2,1980-03-31,1980-03-31,260.0,467,01951910,70000.0,17304,1968-01-02,1986-12-30
3,1980-03-31,1980-03-31,260.0,467,02687410,68250.0,66800,1972-12-14,2009-06-30
4,1980-03-31,1980-03-31,260.0,467,02742910,45000.0,46754,1968-09-09,1990-04-12
...,...,...,...,...,...,...,...,...,...
122358931,2025-12-31,2025-12-31,95105.0,8713,G8588X10,135000.0,22756,2022-01-28,2025-12-31
122358932,2025-12-31,2025-12-31,95105.0,8713,G9611510,184250.0,27119,2025-08-07,2025-12-31
122358933,2025-12-31,2025-12-31,95105.0,8713,L8681T10,28235.0,17782,2018-04-03,2025-12-31
122358934,2025-12-31,2025-12-31,95105.0,8713,M2682V10,65459.0,14928,2014-09-24,2025-12-31


#### Deal with rdate-mgrno-permno duplicates

For the duplicated (rdate, mgrno, permno) observations, I first attempt to match each record using the CRSP name validity window, keeping observations where the reported date sits in between namedt and nameenddt. When no observation in a duplicate group satisfies this condition, I apply a **fallback** rule: I retain observations where the CRSP name window has already ended (nameenddt<rdate) but still lies in the same calendar quarter as the filing date. Among these fallback candidates, I select the observation with the closest prior nameenddt to rdate, ensuring the most temporally consistent stale match is chosen.

In [9]:
%%time
# A subset of duplicates
duptag = holdings_v2.duplicated(subset=['rdate', 'mgrno', 'permno'], keep=False)
dup = holdings_v2.loc[duptag].copy()

# Quarter indicator
dup['rdate'] = pd.to_datetime(dup['rdate'], errors='coerce')
dup['namedt'] = pd.to_datetime(dup['namedt'], errors='coerce')
dup['nameenddt'] = pd.to_datetime(dup['nameenddt'], errors='coerce')
dup['r_quarter'] = dup['rdate'].dt.to_period('Q')
dup['end_quarter'] = dup['nameenddt'].dt.to_period('Q')

# Eligibility condition
dup['eligible'] = (
    ((dup['namedt'] <= dup['rdate']) &
     (dup['rdate'] <= dup['nameenddt']))   # in-window match
    |
    ((dup['nameenddt'] < dup['rdate']) &
     (dup['r_quarter'] == dup['end_quarter']))  # fallback rule
)

# Ranking within each group
def resolve_group(g):
    valid = g[g['eligible']].copy()
    if len(valid) == 0:
        return pd.DataFrame()  # or handle separately if needed
    # closest nameenddt before rdate = max(nameenddt)
    valid = valid.sort_values('nameenddt', ascending=False)
    return valid.iloc[[0]]

# Valid duplicates
dup_valid = (
    dup.set_index(['rdate', 'mgrno', 'permno']).groupby(level=['rdate', 'mgrno', 'permno'], group_keys=False)
       .apply(resolve_group)
       .reset_index()
)
dup_valid

CPU times: user 35 s, sys: 2.18 s, total: 37.2 s
Wall time: 37.3 s


,rdate,mgrno,permno,fdate,NumInst,cusip,shares,namedt,nameenddt,r_quarter,end_quarter,eligible
0,1989-12-31,2730.0,23827,1989-12-31,940,54667610,33100.0,1968-01-02,1990-08-19,1989Q4,1990Q3,True
1,1989-12-31,5850.0,23827,1989-12-31,940,54667610,130900.0,1968-01-02,1990-08-19,1989Q4,1990Q3,True
2,1989-12-31,7800.0,23827,1989-12-31,940,54667610,131212.0,1968-01-02,1990-08-19,1989Q4,1990Q3,True
3,1989-12-31,12120.0,23827,1989-12-31,940,54667610,79635.0,1968-01-02,1990-08-19,1989Q4,1990Q3,True
4,1989-12-31,29800.0,23827,1989-12-31,940,54667610,2560.0,1968-01-02,1990-08-19,1989Q4,1990Q3,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1516,2025-12-31,45495.0,58413,2025-12-31,8713,31374510,133.0,2022-01-03,2025-12-31,2025Q4,2025Q4,True
1517,2025-12-31,45495.0,75844,2025-12-31,8713,67077M10,356.0,2018-01-02,2025-12-31,2025Q4,2025Q4,True
1518,2025-12-31,45495.0,76478,2025-12-31,8713,29631510,689.0,2000-07-11,2025-12-31,2025Q4,2025Q4,True
1519,2025-12-31,45495.0,85414,2025-12-31,8713,03571083,3987.0,2022-09-26,2025-12-31,2025Q4,2025Q4,True


In [10]:
%%time
# Append valid duplicated obs back to the dataset
dup_valid = dup_valid.drop(columns=['r_quarter', 'end_quarter', 'eligible'])
holdings_v2 = pd.concat([holdings_v2.loc[~duptag], dup_valid], ignore_index=True)

# Drop start/end dates
holdings_v2 = holdings_v2.drop(columns=['namedt','nameenddt'])
holdings_v2

CPU times: user 22.5 s, sys: 7.6 s, total: 30.1 s
Wall time: 30.1 s


,rdate,fdate,mgrno,NumInst,cusip,shares,permno
0,1980-03-31,1980-03-31,260.0,467,00080010,50000.0,10006
1,1980-03-31,1980-03-31,260.0,467,01371610,60000.0,24264
2,1980-03-31,1980-03-31,260.0,467,01951910,70000.0,17304
3,1980-03-31,1980-03-31,260.0,467,02687410,68250.0,66800
4,1980-03-31,1980-03-31,260.0,467,02742910,45000.0,46754
...,...,...,...,...,...,...,...
122356792,2025-12-31,2025-12-31,45495.0,8713,31374510,133.0,58413
122356793,2025-12-31,2025-12-31,45495.0,8713,67077M10,356.0,75844
122356794,2025-12-31,2025-12-31,45495.0,8713,29631510,689.0,76478
122356795,2025-12-31,2025-12-31,45495.0,8713,03571083,3987.0,85414


In [11]:
%%time
# Sanity Checks for Duplicates - Ultimately, Should be 0 Duplicates
assert not holdings_v2.duplicated(subset=['rdate', 'mgrno', 'permno']).any(), "Duplicate (rdate, mgrno, permno) observations found!"

CPU times: user 27.2 s, sys: 2.08 s, total: 29.3 s
Wall time: 29.4 s


#### Get prices and shares outstanding

In [12]:
%%time
crsp_m = conn.raw_sql(f"""
                      select permno, mthcaldt, mthret, mthvol, shrout, mthprc
                      from crsp.msf_v2
                      where mthcaldt between '{startdate}' and '{enddate}'
                      and sharetype = 'NS'
                      and securitytype = 'EQTY'
                      and securitysubtype = 'COM'
                      and usincflg = 'Y'
                      and issuertype in ('ACOR','CORP')
                      """, date_cols=['mthcaldt'])

# Change variable format to int
crsp_m[['permno']] = crsp_m[['permno']].astype(int)

# Get month and quarter-end dates
crsp_m['mdate'] = crsp_m['mthcaldt'] + pd.offsets.MonthEnd(0)
crsp_m['qdate'] = crsp_m['mthcaldt'] + pd.offsets.QuarterEnd(0)

# market cap
crsp_m.rename(columns={'mthprc': 'prc'}, inplace=True)
crsp_m['mktcap']  = crsp_m['prc']*crsp_m['shrout']/1e3 # market cap in $mil

# keep only relevant columns
crsp_m = crsp_m[['permno','mdate','qdate','mthcaldt','prc', 'shrout','mktcap']]
crsp_m

CPU times: user 17 s, sys: 1.46 s, total: 18.4 s
Wall time: 23.3 s


,permno,mdate,qdate,mthcaldt,prc,shrout,mktcap
0,10000,1986-01-31,1986-03-31,1986-01-31,4.375,3680,16.1
1,10000,1986-02-28,1986-03-31,1986-02-28,3.25,3680,11.96
2,10000,1986-03-31,1986-03-31,1986-03-31,4.4375,3680,16.33
3,10000,1986-04-30,1986-06-30,1986-04-30,4.0,3793,15.172
4,10000,1986-05-31,1986-06-30,1986-05-30,3.109375,3793,11.793859
...,...,...,...,...,...,...,...
355387,93436,2025-08-31,2025-09-30,2025-08-29,333.87,3225449,1076880.65763
355388,93436,2025-09-30,2025-09-30,2025-09-30,444.72,3324000,1478249.28
355389,93436,2025-10-31,2025-12-31,2025-10-31,456.56,3325819,1518435.92264
355390,93436,2025-11-30,2025-12-31,2025-11-28,430.17,3325819,1430667.55923


In [13]:
# For each stock (permno), each quarter (qdate), find the last monthly date (mdate)
qend = crsp_m[['permno','mdate','qdate']].groupby(['permno','qdate'])['mdate'].max().reset_index()

# Merge back to keep last monthly observation for each quarter
crsp_qend = pd.merge(crsp_m, qend, how='inner', on=['permno','qdate','mdate'])
crsp_qend

,permno,mdate,qdate,mthcaldt,prc,shrout,mktcap
0,10000,1986-03-31,1986-03-31,1986-03-31,4.4375,3680,16.33
1,10000,1986-06-30,1986-06-30,1986-06-30,3.09375,3793,11.734594
2,10000,1986-09-30,1986-09-30,1986-09-30,1.03125,3793,3.911531
3,10000,1986-12-31,1986-12-31,1986-12-31,0.515625,3843,1.981547
4,10000,1987-03-31,1987-03-31,1987-03-31,0.25,3893,0.97325
...,...,...,...,...,...,...,...
967690,93436,2024-12-31,2024-12-31,2024-12-31,403.84,3216517,1298958.22528
967691,93436,2025-03-31,2025-03-31,2025-03-31,259.16,3220000,834495.2
967692,93436,2025-06-30,2025-06-30,2025-06-30,317.66,3224000,1024135.84
967693,93436,2025-09-30,2025-09-30,2025-09-30,444.72,3324000,1478249.28


In [14]:
%%time
# Merge holdings with CRSP
holdings = pd.merge(holdings_v2, crsp_qend[['qdate','permno','prc','shrout','mktcap']], how='inner', left_on=['permno','rdate'], right_on=['permno','qdate'])
holdings = holdings.drop(['qdate','fdate'], axis=1)
holdings

CPU times: user 32.9 s, sys: 9.56 s, total: 42.5 s
Wall time: 42.6 s


,rdate,mgrno,NumInst,cusip,shares,permno,prc,shrout,mktcap
0,1980-03-31,260.0,467,00080010,50000.0,10006,31.875,8859,282.380625
1,1980-03-31,260.0,467,01951910,70000.0,17304,20.25,20522,415.5705
2,1980-03-31,260.0,467,02687410,68250.0,66800,52.5,37698,1979.145
3,1980-03-31,260.0,467,02742910,45000.0,46754,31.5,9547,300.7305
4,1980-03-31,260.0,467,02971710,40000.0,10372,49.25,13751,677.23675
...,...,...,...,...,...,...,...,...,...
90836767,2025-12-31,45495.0,8713,34537086,125816.0,25785,13.12,3913646,51347.03552
90836768,2025-12-31,45495.0,8713,83269640,14530.0,42585,97.81,106694,10435.74014
90836769,2025-12-31,45495.0,8713,61536910,6593.0,48506,510.85,178400,91135.64
90836770,2025-12-31,45495.0,8713,29631510,689.0,76478,195.39,25874,5055.52086


In [15]:
%%time
# Save
holdings.to_parquet(datapath / 'LSEG_1980_2025.parquet', engine='pyarrow')

CPU times: user 28.5 s, sys: 2.25 s, total: 30.7 s
Wall time: 30 s
